# flare — Attention Kernel Benchmark

From-scratch Triton kernels for modern attention variants.

**Requirements:** Colab A100 GPU runtime.

Menu → Runtime → Change runtime type → A100 GPU

## 0. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "❌ No GPU. Runtime → Change runtime type → A100"
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"   SM count: {torch.cuda.get_device_properties(0).multi_processor_count}")
print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Install flare

In [ ]:
import sys, os, subprocess

# Clone the repo
if not os.path.exists('Flare'):
    subprocess.run(['git', 'clone', 'https://github.com/shiloz-stack/Flare.git'])

os.chdir('Flare')
# No pip install needed — just add to Python path
sys.path.insert(0, os.getcwd())

# Verify triton is available
import triton
import triton.language as tl
print(f"✅ Triton {triton.__version__}")
print(f"✅ PyTorch {torch.__version__}")

# Import all kernels
from flare.flash_attn_v2 import flash_attn_v2
from flare.sliding_window import sliding_window_attn
from flare.mla import mla_attention, mla_attention_ref
from flare.kda import kda_attention, kda_attention_ref
print("✅ All flare kernels imported")

## 2. Correctness Tests

Every Triton kernel is compared against a PyTorch reference implementation.
Tolerance: **1e-2** for fp16 (attention involves exp + softmax).

### 2a. FlashAttention v2 vs PyTorch SDPA

In [ ]:
import torch.nn.functional as F

torch.manual_seed(42)
B, H, N, D = 2, 4, 256, 64
q = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
k = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
v = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)

# Non-causal
ref_nc = F.scaled_dot_product_attention(q, k, v, is_causal=False)
tri_nc = flash_attn_v2(q, k, v, causal=False)
err_nc = (tri_nc - ref_nc).abs().max().item()

# Causal
ref_c = F.scaled_dot_product_attention(q, k, v, is_causal=True)
tri_c = flash_attn_v2(q, k, v, causal=True)
err_c = (tri_c - ref_c).abs().max().item()

print(f"FlashAttention v2 — B={B} H={H} N={N} D={D}")
print(f"  Non-causal: max error = {err_nc:.4e}  {'✅ PASS' if err_nc < 1e-2 else '❌ FAIL'}")
print(f"  Causal:     max error = {err_c:.4e}  {'✅ PASS' if err_c < 1e-2 else '❌ FAIL'}")

### 2b. Edge Cases — seq_len not divisible by block size

In [ ]:
print("Edge cases (Br=Bc=64):")
print(f"{'N':>6s}  {'max error':>12s}  {'status':>6s}")
print("-" * 35)

for N_test in [63, 65, 127, 128, 129, 200, 512, 1024]:
    q = torch.randn(1, 2, N_test, 32, device='cuda', dtype=torch.float16)
    k = torch.randn(1, 2, N_test, 32, device='cuda', dtype=torch.float16)
    v = torch.randn(1, 2, N_test, 32, device='cuda', dtype=torch.float16)

    ref = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    tri = flash_attn_v2(q, k, v, causal=True)
    err = (tri - ref).abs().max().item()
    print(f"{N_test:6d}  {err:12.4e}  {'✅' if err < 1e-2 else '❌'}")

### 2c. Sliding-Window Attention

In [ ]:
torch.manual_seed(42)
B, H, N, D, W = 2, 4, 256, 64, 32
q = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
k = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
v = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)

# PyTorch reference (manual loop)
scale = 1.0 / (D ** 0.5)
ref = torch.zeros_like(q)
for b in range(B):
    for h in range(H):
        for i in range(N):
            j_start = max(0, i - W + 1)
            s = (q[b, h, i] @ k[b, h, j_start:i+1].T) * scale
            p = torch.softmax(s, dim=-1)
            ref[b, h, i] = p @ v[b, h, j_start:i+1]

tri = sliding_window_attn(q, k, v, window=W)
err = (tri - ref).abs().max().item()

print(f"Sliding-Window — B={B} H={H} N={N} D={D} W={W}")
print(f"  Max error = {err:.4e}  {'✅ PASS' if err < 1e-2 else '❌ FAIL'}")

### 2d. MLA (Multi-head Latent Attention)

In [ ]:
torch.manual_seed(42)
B, H, N, D = 2, 4, 256, 64
d_compress = 32

q = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
c_kv = torch.randn(B, N, d_compress, device='cuda', dtype=torch.float16)
# Scale weights by 1/sqrt(d_compress) — real models init this way
    w_scale = 1.0 / (d_compress ** 0.5)
    W_UK = torch.randn(d_compress, H, D, device='cuda', dtype=torch.float16) * w_scale
    W_UV = torch.randn(d_compress, H, D, device='cuda', dtype=torch.float16) * w_scale

ref = mla_attention_ref(q, c_kv, W_UK, W_UV)
tri = mla_attention(q, c_kv, W_UK, W_UV)
err = (tri - ref).abs().max().item()

# KV cache comparison
kv_std = 2 * H * D * N * 2  # bytes
kv_mla = d_compress * N * 2

print(f"MLA — B={B} H={H} N={N} D={D} d_compress={d_compress}")
print(f"  KV cache: standard={kv_std//1024}KB  MLA={kv_mla//1024}KB  ({kv_mla/kv_std*100:.1f}%)")
print(f"  Max error = {err:.4e}  {'✅ PASS' if err < 1e-2 else '❌ FAIL'}")

### 2e. KDA (Kimi Delta Attention)

In [ ]:
torch.manual_seed(42)
B, H, N, D = 1, 2, 128, 32
q = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
k = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
v = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
beta = torch.rand(B, H, N, device='cuda', dtype=torch.float16) * 0.5 + 0.5

ref = kda_attention_ref(q, k, v, beta)
print(f"KDA reference output: {ref.shape}")

for chunk in [16, 32, 64]:
    try:
        tri = kda_attention(q, k, v, beta, chunk_size=chunk)
        err = (tri - ref).abs().max().item()
        print(f"  chunk={chunk:3d}: max error = {err:.4e}  {'✅ PASS' if err < 2e-2 else '❌ FAIL'}")
    except Exception as e:
        print(f"  chunk={chunk:3d}: ❌ ERROR: {e}")

## 3. Throughput Benchmark

FlashAttention v2 vs PyTorch SDPA across sequence lengths.
Measures forward-pass latency (ms) on A100-SXM4.

In [ ]:
def bench_kernel(fn, warmup=3, iters=30):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / iters

B, H, D = 1, 8, 64
configs = [512, 1024, 2048, 4096, 8192]

print(f"{'N':>6s} {'flare (ms)':>12s} {'SDPA (ms)':>12s} {'speedup':>8s}")
print("-" * 45)

results = []
for N in configs:
    q = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
    k = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
    v = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)

    t_flare = bench_kernel(lambda: flash_attn_v2(q, k, v, causal=True))
    t_sdpa = bench_kernel(lambda: F.scaled_dot_product_attention(q, k, v, is_causal=True))
    speedup = t_sdpa / t_flare
    results.append((N, t_flare, t_sdpa, speedup))
    print(f"{N:6d} {t_flare:12.3f} {t_sdpa:12.3f} {speedup:7.2f}x")

### Throughput chart

In [ ]:
import matplotlib.pyplot as plt

ns = [r[0] for r in results]
flare_ms = [r[1] for r in results]
sdpa_ms = [r[2] for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Latency
ax1.plot(ns, sdpa_ms, 'o-', label='PyTorch SDPA', color='#2196F3', linewidth=2)
ax1.plot(ns, flare_ms, 's-', label='flare FlashAttention v2', color='#FF5722', linewidth=2)
ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('Latency (ms)')
ax1.set_title('Forward Pass Latency (A100, fp16, causal)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log', base=2)

# Speedup
speedups = [r[3] for r in results]
ax2.bar(range(len(ns)), speedups, color='#4CAF50', alpha=0.8)
ax2.set_xticks(range(len(ns)))
ax2.set_xticklabels([str(n) for n in ns])
ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('Speedup vs SDPA (x)')
ax2.set_title('flare vs PyTorch SDPA')
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('benchmarks/throughput_a100.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to benchmarks/throughput_a100.png")

## 4. MLA KV Cache Savings

In [ ]:
# Compare KV cache sizes: standard MHA vs MLA across seq lengths
# Using DeepSeek-V2-like config: H=128, D=128, d_compress=512

H_config, D_config, d_compress_config = 128, 128, 512
seq_lengths = [1024, 4096, 16384, 65536, 131072]

print(f"Config: H={H_config}, D={D_config}, d_compress={d_compress_config}")
print(f"{'N':>8s} {'Standard KV':>14s} {'MLA KV':>14s} {'Savings':>10s}")
print("-" * 50)

for N in seq_lengths:
    std_bytes = 2 * H_config * D_config * N * 2  # fp16, K+V
    mla_bytes = d_compress_config * N * 2
    savings = (1 - mla_bytes / std_bytes) * 100
    print(f"{N:8d} {std_bytes/1e6:11.1f} MB {mla_bytes/1e6:11.1f} MB {savings:8.1f}%")

## 5. Summary

In [ ]:
print("=" * 60)
print("flare — Benchmark Summary")
print("=" * 60)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}  Triton: {triton.__version__}")
print()
print("Correctness:")
print(f"  FlashAttention v2: {'✅ PASS' if err_nc < 1e-2 and err_c < 1e-2 else '❌ FAIL'}")
print()
print("Throughput (causal, fp16, D=64):")
for N, tf, ts, sp in results:
    print(f"  N={N:5d}: {sp:.2f}x vs SDPA ({tf:.2f}ms vs {ts:.2f}ms)")